# Домашнее задание Feast

2 новые Feature View и 1 On-Demand Feature View.

In [1]:
from feast import FeatureStore
from datetime import datetime
import pandas as pd

store = FeatureStore(repo_path="feature_store/feature_repo")


/Users/elizabeth/pythonProject/.venv/lib/python3.12/site-packages/feast/repo_config.py:420: DeprecationWarning: The serialization version below 3 are deprecated. Specifying `entity_key_serialization_version` to 3 is recommended.
  warnings.warn(


## Пример определений

### Driver Quality Feature View
Содержит признаки качества водителя:
- conv_rate
- acc_rate

### Driver Activity Feature View
Содержит признаки активности:
- avg_daily_trips
- trips_per_week (новый)
- trips_per_month (новый)

### On-Demand Feature View: driver_realtime_metrics
Вычисляет:
- acceptance_gap = acc_rate - conv_rate
- realtime_efficiency = avg_daily_trips * acc_rate


In [2]:
entity_df = pd.DataFrame({
    "driver_id":[1001,1002,1003],
    "event_timestamp":[
        datetime(2021,4,12,10,59,42),
        datetime(2021,4,12,8,12,10),
        datetime(2021,4,12,16,40,26),
    ],
    "request_multiplier":[1.1,1.2,1.3]
})
entity_df


,driver_id,event_timestamp,request_multiplier
0,1001,2021-04-12 10:59:42,1.1
1,1002,2021-04-12 08:12:10,1.2
2,1003,2021-04-12 16:40:26,1.3


## Исторические признаки (training set)

In [3]:
training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "driver_quality_fv:conv_rate",
        "driver_quality_fv:acc_rate",
        "driver_activity_fv:avg_daily_trips",
        "driver_realtime_metrics:acceptance_gap",
        "driver_realtime_metrics:realtime_efficiency",
    ]
).to_df()

training_df.head()


,driver_id,event_timestamp,request_multiplier,conv_rate,acc_rate,avg_daily_trips,acceptance_gap,realtime_efficiency
0,1001,2021-04-12 10:59:42+00:00,1.1,0.709758,0.692957,402,-0.016801,278.568827
1,1002,2021-04-12 08:12:10+00:00,1.2,0.718295,0.584081,370,-0.134214,216.110100
2,1003,2021-04-12 16:40:26+00:00,1.3,0.697411,0.197680,25,-0.499731,4.942010


## Online inference

In [4]:
online_features = store.get_online_features(
    features=[
        "driver_quality_fv:conv_rate",
        "driver_quality_fv:acc_rate",
        "driver_activity_fv:avg_daily_trips",
        "driver_realtime_metrics:acceptance_gap",
        "driver_realtime_metrics:realtime_efficiency",
    ],
    entity_rows=[
        {"driver_id":1001, "request_multiplier":1.1},
        {"driver_id":1002, "request_multiplier":1.2},
    ]
).to_dict()

online_features


/Users/elizabeth/pythonProject/.venv/lib/python3.12/site-packages/feast/infra/key_encoding_utils.py:146: UserWarning: Serialization of entity key with version < 3 is removed. Please use version 3 by setting entity_key_serialization_version=3.To reserializa your online store featrues refer -  https://github.com/feast-dev/feast/blob/master/docs/how-to-guides/entity-reserialization-of-from-v2-to-v3.md
  warnings.warn(


{'driver_id': [1001, 1002],
 'conv_rate': [None, None],
 'acc_rate': [None, None],
 'avg_daily_trips': [None, None],
 'acceptance_gap': [None, None],
 'realtime_efficiency': [None, None]}